[MCP Tools](https://docs.langchain.com/oss/python/langchain/mcp)

| Axis              | Raw SDK            | LangChain           | LangGraph (`create_agent`) |
|---                |---                 |---                  |---                          |
| Boilerplate       | High               | Medium              | Minimal                     |
| Control           | Total              | High                | High (via middleware)       |
| Tool calling      | Hand-roll JSON     | `.bind_tools()`     | Automatic                   |
| State / memory    | Your problem       | Pass list yourself  | Checkpointer + `thread_id`  |
| Streaming         | Raw chunks         | Runnable `.stream`  | Per-node events             |
| Observability     | None built-in      | LangSmith compat.   | LangSmith native            |
| Human-in-loop     | DIY                | DIY                 | `interrupt()` primitive     |

These are the rules to actually apply in real project kickoffs:

1. **Use the raw SDK when** your code path is a single LLM call with no tool use, no retry logic, and no downstream pipeline. Think: a one-shot classifier or a prompt in a cron job. Every abstraction you add is a line you could delete.

2. **Use LangChain (LCEL / primitives, no graph) when** your dataflow is a pure DAG: prompt → model → parser, or a fan-out/fan-in with `RunnableParallel`. If drawing it looks like a tree with no loops and no branching-on-state, you don't need a graph.

3. **Use LangGraph `create_agent` when** you want a tool-using agent and the default ReAct loop is enough. Stop there; don't build a custom graph until you feel friction.

4. **Use a custom LangGraph `StateGraph` when** *any* of: you have a loop bounded by a counter, routing based on LLM-evaluated content, human-in-the-loop approval, or multiple agents coordinating on shared state.

5. **Always add LangSmith tracing** as soon as the code path exceeds a single LLM call — even for LangChain-only code. Cost is zero; future debugging value is large.


# LangChain & LangGraph — Short Notes

> **Goal:** Build production-grade LLM applications with tools, structured output, state, memory, routing, retries, persistence, observability, and human-in-the-loop workflows.

---

# 1. LangChain vs LangGraph vs Raw SDK

## Raw LLM SDK

Use the raw SDK when the application is simple:

```text
User
  ↓
LLM
  ↓
Response

|                         | LangChain                 | LangGraph                      |
| ----------------------- | ------------------------- | ------------------------------ |
| Main purpose            | Build LLM applications    | Build agent/workflow execution |
| Abstraction             | Components + integrations | Graph/state/execution          |
| Simple LLM calls        | ✅                         | Usually unnecessary            |
| Prompt → LLM → parser   | ✅                         | Overkill                       |
| RAG                     | ✅                         | Can use LangChain components   |
| Tool calling            | ✅                         | ✅                              |
| Multiple steps          | ✅                         | ✅                              |
| Branching               | Limited/simple            | ✅                              |
| Loops                   | Limited                   | ✅                              |
| Stateful agents         | Not its main strength     | ✅                              |
| Multi-agent workflows   | Possible                  | ✅                              |
| Human-in-the-loop       | Possible                  | Strong support                 |
| Persistence/checkpoints | Not the core              | ✅                              |


| Syntax | Used For |
|---|---|
| `init_chat_model()` | Initialize a chat/LLM model |
| `model.invoke()` | Execute the model once |
| `HumanMessage` | Represents user input |
| `AIMessage` | Represents AI response |
| `SystemMessage` | Defines model behavior/instructions |
| `ToolMessage` | Represents a tool's result |
| `@tool` | Convert a Python function into a LangChain tool |
| `@tool(args_schema=...)` | Give a tool a custom Pydantic input schema |
| `.bind_tools()` | Make tools available to the LLM |
| `.tool_calls` | Read tools requested by the LLM |
| `.with_structured_output()` | Force/parse model output into a defined schema |
| `BaseModel` | Define a structured/Pydantic schema |
| `Field()` | Add validation or descriptions to schema fields |
| `Literal[...]` | Restrict a field to specific allowed values |

| Syntax            | Used For                                  |
| ----------------- | ----------------------------------------- |
| `create_agent()`  | Create a standard tool-using agent        |
| `ToolNode`        | Execute tools inside a LangGraph          |
| `tools_condition` | Decide whether to execute tools or finish |
| `.bind_tools()`   | Make tools available to the model         |


| Syntax                | Used For                      |
| --------------------- | ----------------------------- |
| `get_state()`         | Get the latest state snapshot |
| `get_state_history()` | View previous checkpoints     |
| `snapshot.values`     | Read current state values     |
| `snapshot.next`       | See what executes next        |
| `snapshot.metadata`   | Inspect execution metadata    |


| Syntax                      | Remember It As              |
| --------------------------- | --------------------------- |
| `init_chat_model()`         | Initialize model            |
| `HumanMessage`              | User message                |
| `AIMessage`                 | AI message                  |
| `SystemMessage`             | System instruction          |
| `ToolMessage`               | Tool result                 |
| `@tool`                     | Create tool                 |
| `bind_tools()`              | Give model tools            |
| `tool_calls`                | Read requested tools        |
| `with_structured_output()`  | Structured LLM result       |
| `BaseModel`                 | Define schema               |
| `Field()`                   | Validate/describe field     |
| `prompt \| model \| parser` | LCEL pipeline               |
| `StateGraph()`              | Create custom graph         |
| `add_node()`                | Add processing step         |
| `add_edge()`                | Fixed route                 |
| `add_conditional_edges()`   | Conditional route           |
| `Command()`                 | Update + route              |
| `START`                     | Entry                       |
| `END`                       | Exit                        |
| `ToolNode`                  | Execute tools               |
| `tools_condition`           | Tool/no-tool routing        |
| `Annotated[..., reducer]`   | Define state merge behavior |
| `add_messages`              | Merge message state         |
| `checkpointer`              | Persist state               |
| `thread_id`                 | Identify conversation       |
| `get_state()`               | Current state               |
| `get_state_history()`       | Past checkpoints            |
| `stream()`                  | Stream execution            |
| `interrupt()`               | Pause for human             |
| `Command(resume=...)`       | Resume after HITL           |
| `create_agent()`            | Standard tool agent         |
